# Marigold V2 — photo folder to reconstruction evidence

Run depth, normals, albedo and see-through depth over selected photos. Export the
**raw float32 `.npy` arrays**, lossless PNG previews, prepared/original photos,
source mappings, transforms, model fingerprints and settings in one ZIP.

This notebook is self-contained: uploading this `.ipynb` is enough. It uses the
pinned official Marigold CLI; it does not need the house-reconstruction repository.
GPU inference requires a **Linux NVIDIA/CUDA machine**. The upstream estimate is
about 17 GB VRAM at 1024²; a 24 GB+ GPU is a practical starting point. The Mac can
inspect the notebook/results but cannot run this CUDA inference. No paid GPU is
provisioned and nothing is uploaded by the notebook.

**Environment:** preferably use the upstream Python 3.10 conda environment. On a
fresh GPU machine, prepare it in a terminal, then select its Jupyter kernel:

```bash
git clone https://github.com/huawei-bayerlab/marigold-v2.git ~/marigold-v2
cd ~/marigold-v2
git checkout cc6a7031abcd59fd9e1ceff7fdd0d9687d389bc5
bash setup/setup_env.sh marigold-v2 cu128
conda activate marigold-v2
python -m pip install jupyterlab ipykernel
python -m ipykernel install --user --name marigold-v2 --display-name "Marigold V2"
jupyter lab
```

Alternatively, the optional install cell below installs into a compatible existing
Linux GPU notebook kernel. Restart that kernel after installing. Copy/mount your
photo directory onto that GPU machine first (for example a mounted Drive folder).

[Upstream instructions](https://github.com/huawei-bayerlab/marigold-v2#quick-start)
· [CLI source](https://github.com/huawei-bayerlab/marigold-v2/blob/cc6a7031abcd59fd9e1ceff7fdd0d9687d389bc5/scripts/infer.py)


## 1. Configuration

Change `PHOTO_DIR` and `OUTPUT_DIR`. Keep the two directories separate. Put a
persistent output directory on a mounted disk if your GPU session is temporary.
A completed result is reused only when its file hashes and array checks pass.
Changing inputs, settings, model assets or environment requires a new output folder.


In [ ]:
from pathlib import Path

PHOTO_DIR = Path("/path/to/photos")
OUTPUT_DIR = Path("/path/to/marigold-results/broom-road-run-01")
PROPERTY_ID = "3broomroad"
REPO_DIR = Path.home() / "marigold-v2"
ASSETS_DIR = REPO_DIR / "assets"
MODALITIES = ["depth", "normals", "albedo", "depth_seethrough"]
MAX_EDGE = 1024  # Preserve aspect ratio; do not upscale. Lower if VRAM is tight.
SEED = 2025
INSTALL_DEPENDENCIES = False  # Enable only for a fresh Linux GPU kernel, then restart.
DOWNLOAD_MODELS = True       # Only selected inference weights; resumes existing downloads.
DISK_RESERVE_GIB = 8          # Working headroom, in addition to missing model bytes.
REMOVE_UNUSED_ASSETS = False # Opt-in cleanup of unused weights from the old broad downloader.
CLEAN_COMPLETED_ATTEMPTS = True # Keep results/logs/configs; remove redundant CLI image copies.
INCLUDE_ORIGINALS_IN_ZIP = True

# By default skip previous prediction previews and obvious plans/aerials.
# Review the inventory below; names cannot reliably classify photo content.
EXCLUDE_PATTERNS = ["*_depth*", "*_normal*", "*_albedo*", "*seethrough*",
                    "*floorplan*", "*floor_plan*", "aerial.*"]
EXCLUDE_RELATIVE_PATHS = []   # Exact paths shown in the inventory, e.g. "misc/map.jpg".
ONLY_RELATIVE_PATHS = []      # Empty = all eligible photos, including interiors.


## 2. Pinned code and optional kernel setup

A new checkout is pinned automatically. An existing checkout is never reset.
The optional installation uses the upstream Torch versions and project dependencies;
it must run in an isolated GPU notebook environment, not your application environment.


In [ ]:
import os, platform, subprocess, sys
PINNED_COMMIT = "cc6a7031abcd59fd9e1ceff7fdd0d9687d389bc5"
REPO_DIR, ASSETS_DIR = REPO_DIR.expanduser().resolve(), ASSETS_DIR.expanduser().resolve()
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/huawei-bayerlab/marigold-v2.git", str(REPO_DIR)], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", PINNED_COMMIT], check=True)
head = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
if head != PINNED_COMMIT:
    raise RuntimeError("Existing checkout differs. Set REPO_DIR to a new folder for the pinned version.")
if INSTALL_DEPENDENCIES:
    if platform.system() != "Linux":
        raise RuntimeError("Use a Linux CUDA GPU kernel for dependency installation.")
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-cache-dir", "torch==2.10.0", "torchvision==0.25.0",
                    "--index-url", "https://download.pytorch.org/whl/cu128"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-cache-dir", "-e", str(REPO_DIR)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "check"], check=True)
    raise RuntimeError("Installation complete. Restart the kernel, set INSTALL_DEPENDENCIES=False, then run again.")
print("Upstream revision:", head)


## 3. Folder-processing helpers

This cell is embedded so the notebook can travel by itself. The matching
`marigold_batch.py` in this repository is the tested source. Resume is per
photo/modality; each remaining modality batch loads its model once. An interrupted
batch salvages valid outputs; rerunning processes missing or damaged results.


In [ ]:
HELPER_SHA256 = '423bbb18a15ab5eb712d5cda3165ec26012dac6282b2e861e8ad4c106b7c3252'
"""Notebook helpers: prepare sources, run official CLI, validate and export evidence.
Only NumPy/Pillow are needed for preparation and tests. Inference uses a separate
CLI process in the active Marigold environment. No reconstruction state is changed.
"""
from pathlib import Path
from datetime import datetime, timezone
import fnmatch
import hashlib
import json
import os
import shutil
import signal
import subprocess
import sys
import uuid
import zipfile
import numpy as np
from PIL import Image, ImageOps

UPSTREAM_COMMIT = 'cc6a7031abcd59fd9e1ceff7fdd0d9687d389bc5'
TASKS = {
    'depth': {'modality': 'depth', 'checkpoint': 'depth/Log-stage2', 'preview': 'depth_spectral', 'encoding': 'affine-invariant log depth; not metres'},
    'normals': {'modality': 'normals', 'checkpoint': 'normals', 'preview': 'normals', 'encoding': 'camera-space XYZ vectors, CHW; renormalize after interpolation if used downstream'},
    'albedo': {'modality': 'albedo', 'checkpoint': 'albedo', 'preview': 'albedo', 'encoding': 'sRGB via upstream gamma-2.2 conversion, CHW; not linear RGB'},
    'depth_seethrough': {'modality': 'depth', 'checkpoint': 'depth/Log-layered', 'preview': 'depth_spectral', 'encoding': 'see-through affine-invariant log depth; separate from exterior surface depth'},
}
DEFAULT_EXCLUDES = ['*_depth*', '*_normal*', '*_albedo*', '*seethrough*', '*floorplan*', '*floor_plan*', 'aerial.*']

# File selections verified against the pinned inference config/loader, which load
# only VAE + transformer and precomputed prompt embeddings (no text encoder).
MODEL_REVISIONS = {
    'Qwen/Qwen-Image-Edit-2509': 'd3968ef930e841f4c73640fb8afa3b306a78167e',
    'huawei-bayerlab/marigold-v2-0': '6fd6d1ca246c9d2d99a4d8ac375a4eccc87178ad',
}
EMBED_PREFIXES = {'depth': 'qwen_edit_2509_qwen_depth_realimg512',
                  'normals': 'qwen_edit_2509_qwen_normals_dummy512',
                  'albedo': 'qwen_edit_2509_qwen_albedo_rgb_dummy512'}


def asset_specs(modalities):
    if not modalities or any(m not in TASKS for m in modalities):
        raise ValueError('Select supported modalities before downloading assets.')
    patterns = [TASKS[m]['checkpoint'] + '/*' for m in modalities]
    patterns += ['qwen_text_embeddings/' + EMBED_PREFIXES[m] + suffix
                 for m in sorted({TASKS[m]['modality'] for m in modalities})
                 for suffix in ['_prompt_embeds.pt', '_prompt_mask.pt']]
    return [dict(repo_id=repo, revision=MODEL_REVISIONS[repo], directory=directory, patterns=pats)
            for repo, directory, pats in [
                ('Qwen/Qwen-Image-Edit-2509', 'Qwen-Image-Edit-2509', ['transformer/*', 'vae/*']),
                ('huawei-bayerlab/marigold-v2-0', 'Marigold-V2', patterns)]]


def plan_assets(assets, modalities, api=None):
    """Read metadata only; estimate missing bytes conservatively, including partial files."""
    if api is None:
        from huggingface_hub import HfApi
        api = HfApi()
    assets = Path(assets).resolve()
    specs = asset_specs(modalities)
    for spec in specs:
        info = api.model_info(spec['repo_id'], revision=spec['revision'], files_metadata=True)
        files = []
        for entry in info.siblings:
            name = entry.rfilename
            if not any(fnmatch.fnmatch(name, p) for p in spec['patterns']):
                continue
            if Path(name).is_absolute() or '..' in Path(name).parts or entry.size is None:
                raise ValueError('Invalid or missing remote file metadata: ' + name)
            target = assets/'checkpoints'/spec['directory']/name
            files.append({'name': name, 'size': entry.size,
                          'missing_bytes': 0 if target.is_file() and target.stat().st_size == entry.size else entry.size})
        if any(not any(fnmatch.fnmatch(f['name'], p) for f in files) for p in spec['patterns']):
            raise ValueError('Required model files absent from pinned metadata: ' + spec['repo_id'])
        spec['files'] = files
    return specs


def download_inference_assets(assets, modalities, reserve_gib=8, api=None, download=None):
    """Download only inference assets, directly to local_dir, after a disk preflight."""
    if reserve_gib < 0:
        raise ValueError('Disk reserve must be nonnegative.')
    assets = Path(assets).resolve()
    assets.mkdir(parents=True, exist_ok=True)
    specs = plan_assets(assets, modalities, api)
    required = sum(f['missing_bytes'] for s in specs for f in s['files'])
    free = shutil.disk_usage(assets).free
    reserve = int(reserve_gib * 2**30)
    print(f'Assets disk: {free/2**30:.1f} GiB free; up to {required/2**30:.1f} GiB to download; '
          f'{reserve_gib:g} GiB working reserve.', flush=True)
    if required + reserve > free:
        raise OSError('Insufficient disk for selected weights plus working reserve. '
                      'Review unused assets/package cache or choose a larger ASSETS_DIR disk. '
                      'Reducing the photo batch does not reduce model storage. No weights downloaded.')
    if download is None:
        from huggingface_hub import snapshot_download
        download = snapshot_download
    for spec in specs:
        print('Downloading/resuming:', spec['repo_id'], flush=True)
        download(repo_id=spec['repo_id'], revision=spec['revision'],
                 local_dir=str(assets/'checkpoints'/spec['directory']),
                 allow_patterns=[f['name'] for f in spec['files']], max_workers=1)
    return specs


def prune_unused_assets(assets, modalities, dry_run=True):
    """Opt-in recovery for a dedicated inference asset directory; never touch sources/results.

    Stop all jobs using this directory first. Pruning may invalidate earlier full
    asset locks: retain those assets if another run depends on them.
    """
    assets = Path(assets).resolve()
    specs = asset_specs(modalities)
    qwen = assets/'checkpoints'/'Qwen-Image-Edit-2509'
    marigold = assets/'checkpoints'/'Marigold-V2'
    candidates = [qwen/name for name in ['text_encoder', 'tokenizer', 'processor', 'scheduler']]
    selected = {TASKS[m]['checkpoint'] for m in modalities}
    candidates += [marigold/name for name in ['depth/Disparity-base', 'depth/Disparity-layered',
                   'depth/Log-stage1', 'depth/Log-stage2', 'depth/Log-layered',
                   'depth/Uniform-base', 'depth/Uniform-layered', 'normals', 'albedo'] if name not in selected]
    # local_dir keeps interrupted transfers under .cache/huggingface/download.
    # Remove only caches for the explicitly unused component directories above.
    candidates += [root/'.cache'/'huggingface'/'download'/p.relative_to(root)
                   for p in list(candidates) for root in [qwen, marigold] if p.is_relative_to(root)]
    for path in (marigold/'qwen_text_embeddings').glob('*.pt'):
        if not any(fnmatch.fnmatch(path.relative_to(marigold).as_posix(), p) for p in specs[1]['patterns']):
            candidates.append(path)
    report = []
    for path in candidates:
        # Do not follow links into other model stores or mount points.
        if not path.exists() or path.is_symlink() or not path.resolve().is_relative_to(assets):
            continue
        size = sum(p.stat().st_size for p in path.rglob('*') if p.is_file() and not p.is_symlink()) if path.is_dir() else path.stat().st_size
        report.append({'path': str(path), 'bytes': size})
        if not dry_run:
            shutil.rmtree(path) if path.is_dir() else path.unlink()
    return report


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(4 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(json.dumps(value, indent=2, allow_nan=False) + '\n')
    temp.replace(path)


def discover(photo_dir, excludes=DEFAULT_EXCLUDES):
    root = Path(photo_dir).expanduser().resolve()
    if not root.is_dir():
        raise ValueError(f'Photo folder does not exist: {root}')
    found, skipped = [], []
    for path in sorted(root.rglob('*')):
        if not path.is_file() or path.suffix.lower() not in {'.jpg', '.jpeg', '.png', '.webp', '.tif', '.tiff'}:
            continue
        rel = path.relative_to(root).as_posix()
        if not path.resolve().is_relative_to(root):
            raise ValueError(f'Source symlink escapes photo folder: {rel}')
        if any(part.startswith('.') for part in Path(rel).parts) or any(fnmatch.fnmatch(rel.lower(), p.lower()) or fnmatch.fnmatch(path.name.lower(), p.lower()) for p in excludes):
            skipped.append({'source': rel, 'reason': 'excluded by filename pattern'})
            continue
        try:
            with Image.open(path) as im:
                if getattr(im, 'n_frames', 1) != 1:
                    raise ValueError('Multi-frame image: supply one photo per file')
                original_size, orientation = list(im.size), int(im.getexif().get(274, 1))
                rgb = ImageOps.exif_transpose(im).convert('RGB')
                pixel_hash = hashlib.sha256(str(rgb.size).encode() + rgb.tobytes()).hexdigest()
            found.append({'source': rel, 'source_sha256': sha256(path), 'original_size': original_size, 'exif_orientation': orientation, 'oriented_size': list(rgb.size), 'pixel_sha256': pixel_hash})
        except Exception as exc:
            skipped.append({'source': rel, 'reason': 'unreadable: ' + str(exc)})
    return found, skipped


def prepare(photo_dir, output_dir, selected, skipped, modalities, max_edge=1024, seed=2025, property_id='property'):
    root, out = Path(photo_dir).expanduser().resolve(), Path(output_dir).expanduser().resolve()
    if out == root or out.is_relative_to(root) or root.is_relative_to(out):
        raise ValueError('Keep photo and output folders separate, with neither inside the other.')
    if len({s['source'] for s in selected}) != len(selected):
        raise ValueError('Source selection contains duplicate paths.')
    if not selected or not modalities or len(set(modalities)) != len(modalities) or any(m not in TASKS for m in modalities):
        raise ValueError('Select photos and unique, supported modalities.')
    if not isinstance(max_edge, int) or max_edge < 64:
        raise ValueError('MAX_EDGE must be an integer of at least 64 pixels.')
    config = {'property_id': property_id, 'modalities': list(modalities), 'max_edge': max_edge, 'seed': int(seed), 'source_root': str(root), 'sources': selected}
    fingerprint = hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()
    manifest_path = out / 'manifest.json'
    if manifest_path.exists():
        manifest = json.loads(manifest_path.read_text())
        if manifest.get('input_fingerprint') != fingerprint:
            raise ValueError('Inputs/settings changed. Choose a new OUTPUT_DIR to preserve the earlier run.')
        for row in manifest['images']:
            if sha256(out / row['input']) != row['input_sha256'] or sha256(root / row['source']) != row['source_sha256']:
                raise ValueError('Prepared input or original source changed; choose a new run folder.')
        return manifest
    if out.exists() and any(out.iterdir()):
        raise ValueError('OUTPUT_DIR is not empty and has no manifest. Choose a new directory.')
    out.mkdir(parents=True, exist_ok=True)
    manifest = {'schemaVersion': 1, 'kind': 'monocular-prediction-evidence', 'status': 'prepared', 'input_fingerprint': fingerprint, 'settings': config, 'created_at': datetime.now(timezone.utc).isoformat(), 'images': [], 'skipped': skipped, 'tasks': TASKS, 'attempts': [], 'environment': None}
    seen = {}
    for source in selected:
        path = (root / source['source']).resolve()
        if not path.is_relative_to(root) or sha256(path) != source['source_sha256']:
            raise ValueError('Source path or content changed during preparation.')
        if source['pixel_sha256'] in seen:
            seen[source['pixel_sha256']]['aliases'].append(dict(source))
            continue
        # Content IDs avoid same-stem collisions and keep duplicate aliases explicit.
        id = source['pixel_sha256'][:24]
        with Image.open(path) as im:
            rgb = ImageOps.exif_transpose(im).convert('RGB')
            rgb.thumbnail((max_edge, max_edge), Image.Resampling.LANCZOS)
            target = out / 'inputs' / (id + '.png')
            target.parent.mkdir(exist_ok=True)
            rgb.save(target)
        row = {**source, 'id': id, 'aliases': [], 'input': target.relative_to(out).as_posix(), 'input_sha256': sha256(target), 'input_size': list(rgb.size), 'transform': {'exif_transposed': True, 'rgb_conversion': True, 'crop': None, 'scale_xy': [rgb.width/source['oriented_size'][0], rgb.height/source['oriented_size'][1]]}, 'outputs': {}}
        manifest['images'].append(row)
        seen[source['pixel_sha256']] = row
    write_json(manifest_path, manifest)
    return manifest


def validate_output(npy, png, modality, size):
    arr = np.load(npy, allow_pickle=False)
    expected = (size[1], size[0]) if TASKS[modality]['modality'] == 'depth' else (3, size[1], size[0])
    if arr.shape != expected or arr.dtype != np.float32 or not np.isfinite(arr).all():
        raise ValueError(f'Invalid array: expected finite float32 {expected}, got {arr.dtype} {arr.shape}')
    with Image.open(png) as im:
        im.load()
        if im.format != 'PNG' or im.size != tuple(size):
            raise ValueError('Preview must be a PNG aligned with its prepared source.')
    stats = {'shape': list(arr.shape), 'dtype': str(arr.dtype), 'minimum': float(arr.min()), 'maximum': float(arr.max()), 'std': float(arr.std()), 'warnings': []}
    if stats['std'] < 1e-7:
        stats['warnings'].append('Nearly constant prediction: inspect visually')
    if modality == 'normals':
        lengths = np.linalg.norm(arr, axis=0)
        stats['median_normal_length'] = float(np.median(lengths))
        stats['zero_normal_fraction'] = float(np.mean(lengths < 1e-6))
        if stats['zero_normal_fraction'] > .01 or np.median(lengths) < .9 or np.median(lengths) > 1.1:
            stats['warnings'].append('Unusual normal magnitudes: inspect raw vectors')
    if modality == 'albedo' and (arr.min() < -.01 or arr.max() > 1.01):
        stats['warnings'].append('Albedo outside expected display range')
    return stats


def output_valid(out, row, modality):
    record = row['outputs'].get(modality)
    if not record:
        return False
    try:
        for key in ['npy', 'png']:
            if sha256(out / record[key]) != record[key + '_sha256']:
                return False
        validate_output(out / record['npy'], out / record['png'], modality, row['input_size'])
        return True
    except (OSError, ValueError):
        return False


QWEN_API_CHECK = '''import inspect
import diffusers
from diffusers import QwenImageTransformer2DModel
if diffusers.__version__ != '0.38.0':
    raise RuntimeError(f"Marigold requires diffusers==0.38.0; found {diffusers.__version__}. "
                       "Install the pinned version in the inference interpreter.")
signature = inspect.signature(QwenImageTransformer2DModel.forward)
signature.bind_partial(self=None, hidden_states=None, timestep=None,
                       encoder_hidden_states=None, encoder_hidden_states_mask=None,
                       img_shapes=None, txt_seq_lens=None, guidance=None,
                       attention_kwargs=None, return_dict=False)
print('Diffusers 0.38.0 Qwen forward API check passed (no weights loaded).')
'''


def check_inference_imports(repo):
    """Test the actual CLI interpreter's imports before downloading/loading weights."""
    script = QWEN_API_CHECK + (
        "import importlib\n"
        "from omegaconf import OmegaConf\n"
        "cfg = OmegaConf.load('evaluation/config/inference_depth.yaml')\n"
        "for name in list(cfg.register_modules) + ['marigoldv2.validation.folder_steps']:\n"
        "    importlib.import_module(name)\n"
        "print('Inference imports passed (model execution not tested).')\n"
    )
    result = subprocess.run([sys.executable, '-c', script], cwd=Path(repo),
                            text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout, flush=True)
    if result.returncode:
        raise RuntimeError('Inference dependency check failed in ' + sys.executable +
                           '. Fix the import error above before downloading weights or running step 6.')


def print_attempt_failure(attempt, record, lines=80):
    print(f"Inference failed: {record['modality']} (exit {record.get('returncode')}). "
          f"{record.get('error', '')}", flush=True)
    log = Path(attempt)/'inference.log'
    if log.exists():
        # Bound memory even when model loaders produce very large logs.
        from collections import deque
        with log.open(errors='replace') as stream:
            print(''.join(deque(stream, maxlen=lines)), flush=True)
    print('Full log:', log, flush=True)


def runtime_provenance(repo, assets, modalities, out):
    repo, assets, out = Path(repo).resolve(), Path(assets).resolve(), Path(out).resolve()
    commit = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
    if commit != UPSTREAM_COMMIT:
        raise ValueError(f'Expected reviewed upstream revision {UPSTREAM_COMMIT}, got {commit}.')
    if subprocess.check_output(['git', '-C', str(repo), 'diff', 'HEAD', '--'], text=True).strip():
        raise ValueError('Upstream tracked files have local modifications; use a clean pinned checkout.')
    # Freeze original asset paths before inference adds any quantisation caches.
    lock = out / 'asset-lock.json'
    existing_lock = lock.exists()
    if existing_lock:
        locked = json.loads(lock.read_text())
    else:
        roots = [assets/'checkpoints'/'Qwen-Image-Edit-2509', assets/'checkpoints'/'Marigold-V2'/'qwen_text_embeddings']
        roots += [assets/'checkpoints'/'Marigold-V2'/TASKS[m]['checkpoint'] for m in modalities]
        paths = sorted({p for r in roots for p in r.rglob('*') if p.is_file() and '.cache' not in p.relative_to(assets).parts})
        if any(not r.is_dir() for r in roots) or not paths:
            raise ValueError('Assets missing. Run the notebook download cell first.')
        locked = {'files': {p.relative_to(assets).as_posix(): sha256(p) for p in paths}}
        write_json(lock, locked)
    for name, digest in (locked['files'].items() if existing_lock else []):
        if sha256(assets / name) != digest:
            raise ValueError(f'Model asset changed: {name}. Use a new run folder.')
    packages = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
    import torch
    return {'upstream_commit': commit, 'asset_lock_sha256': sha256(lock), 'python': sys.version, 'packages': packages, 'torch': torch.__version__, 'cuda': torch.version.cuda, 'gpu': torch.cuda.get_device_name(0), 'helper_sha256': HELPER_SHA256 if 'HELPER_SHA256' in globals() else sha256(__file__)}


def run_batch(repo, assets, output_dir, provenance, execute=None, clean_completed_attempts=True):
    """Resume per image/modality. One fresh subprocess per pending modality batch."""
    out, repo, assets = Path(output_dir).resolve(), Path(repo).resolve(), Path(assets).resolve()
    manifest = json.loads((out / 'manifest.json').read_text())
    if manifest['environment'] is not None and manifest['environment'] != provenance:
        raise ValueError('Code, environment or assets changed. Choose a new OUTPUT_DIR.')
    manifest['environment'] = provenance
    env = {**os.environ, 'DEPTH_ASSETS_DIR': str(assets), 'PYTHONUNBUFFERED': '1'}
    for row in manifest['images']:
        if sha256(out / row['input']) != row['input_sha256']:
            raise ValueError('Prepared input changed.')
    for modality in manifest['settings']['modalities']:
        pending = [r for r in manifest['images'] if not output_valid(out, r, modality)]
        if not pending:
            print(modality + ': all validated outputs already present', flush=True)
            continue
        attempt = out / 'attempts' / (modality + '-' + uuid.uuid4().hex[:12])
        inputs, outputs = attempt/'inputs', attempt/'outputs'
        inputs.mkdir(parents=True)
        for row in pending:
            target = inputs / (row['id'] + '.png')
            try:
                os.link(out / row['input'], target)
            except OSError:
                shutil.copyfile(out / row['input'], target)
        command = [sys.executable, str(repo/'scripts'/'infer.py'), '--modality', TASKS[modality]['modality'], '--checkpoint', str(assets/'checkpoints'/'Marigold-V2'/TASKS[modality]['checkpoint']), '--image_dir', str(inputs), '--output_dir', str(outputs), '--seed', str(manifest['settings']['seed'])]
        record = {'modality': modality, 'command': command, 'directory': attempt.relative_to(out).as_posix(), 'status': 'running', 'started_at': datetime.now(timezone.utc).isoformat()}
        manifest['attempts'].append(record)
        manifest['status'] = 'running'
        write_json(out/'manifest.json', manifest)
        print(f'{modality}: {len(pending)} pending photos. Log: {attempt / "inference.log"}', flush=True)
        interrupted = False
        try:
            if execute is None:
                with (attempt/'inference.log').open('w') as log:
                    process = subprocess.Popen(command, cwd=repo, env=env, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
                    try:
                        record['returncode'] = process.wait()
                    except KeyboardInterrupt:
                        # infer.py launches the evaluator as a child. Stop the process tree.
                        try:
                            os.killpg(process.pid, signal.SIGTERM)
                        except ProcessLookupError:
                            pass
                        try:
                            process.wait(timeout=10)
                        except subprocess.TimeoutExpired:
                            os.killpg(process.pid, signal.SIGKILL)
                            process.wait()
                        raise
            else:
                record['returncode'] = execute(command, repo, env, attempt/'inference.log')
        except KeyboardInterrupt:
            interrupted = True
            record['returncode'] = None
        except Exception as exc:
            record['returncode'] = None
            record['error'] = str(exc)
        record['status'] = 'complete' if record['returncode'] == 0 else 'interrupted' if interrupted else 'failed'
        record['finished_at'] = datetime.now(timezone.utc).isoformat()
        for row in pending:
            npy = outputs/'images'/'predictions_npy'/(row['id']+'.npy')
            png = outputs/'images'/'visualizations'/TASKS[modality]['preview']/(row['id']+'.png')
            try:
                stats = validate_output(npy, png, modality, row['input_size'])
                target = out/'results'/row['id']
                target.mkdir(parents=True, exist_ok=True)
                dest_npy, dest_png = target/(modality+'.npy'), target/(modality+'.png')
                shutil.copyfile(npy, dest_npy)
                shutil.copyfile(png, dest_png)
                row['outputs'][modality] = {'npy': dest_npy.relative_to(out).as_posix(), 'png': dest_png.relative_to(out).as_posix(), 'npy_sha256': sha256(dest_npy), 'png_sha256': sha256(dest_png), 'encoding': TASKS[modality]['encoding'], 'stats': stats, 'attempt': record['directory']}
                row.pop('last_error', None)
            except Exception as exc:
                row['outputs'].pop(modality, None)
                row['last_error'] = modality + ': ' + str(exc)
        manifest['status'] = 'partial'
        write_json(out/'manifest.json', manifest)
        # Persist and verify results before removing disposable CLI copies.
        # Retain failed attempts for diagnosis and preserve logs/configs always.
        if clean_completed_attempts and record['status'] == 'complete' and all(output_valid(out, r, modality) for r in pending):
            for folder in [inputs, outputs]:
                for path in folder.rglob('*'):
                    if path.is_file() and path.suffix not in {'.yaml', '.yml', '.log'}:
                        path.unlink()
            record['temporary_images_removed'] = True
            write_json(out/'manifest.json', manifest)
        if interrupted:
            raise KeyboardInterrupt('Interrupted; validated outputs saved. Rerun this cell to resume.')
        validated = sum(output_valid(out, row, modality) for row in pending)
        if record['returncode'] != 0 or validated != len(pending):
            print_attempt_failure(attempt, record)
        if validated == 0:
            print('No validated predictions in this attempt; stopping before the next modality. '
                  'Resolve the log error above, then retry.', flush=True)
            break
    manifest['status'] = 'complete' if all(output_valid(out, r, m) for r in manifest['images'] for m in manifest['settings']['modalities']) else 'partial'
    write_json(out/'manifest.json', manifest)
    return manifest


def export_zip(output_dir, include_originals=True):
    out = Path(output_dir).resolve()
    manifest = json.loads((out/'manifest.json').read_text())
    if manifest['status'] != 'complete' or not all(output_valid(out, r, m) for r in manifest['images'] for m in manifest['settings']['modalities']):
        raise ValueError('Run is incomplete or outputs changed. Resolve failures before final export.')
    archive = out.parent / (out.name + '-evidence-' + uuid.uuid4().hex[:8] + '.zip')
    temporary = archive.with_suffix('.zip.partial')
    with zipfile.ZipFile(temporary, 'w', compression=zipfile.ZIP_DEFLATED, allowZip64=True) as z:
        for name in ['manifest.json', 'asset-lock.json', 'download-plan.json']:
            if (out/name).exists():
                z.write(out/name, name)
        for folder in ['inputs', 'results']:
            for path in sorted((out/folder).rglob('*')):
                if path.is_file():
                    z.write(path, path.relative_to(out).as_posix())
        for path in sorted((out/'attempts').rglob('*')):
            if path.is_file() and (path.name == 'inference.log' or path.suffix == '.yaml'):
                z.write(path, path.relative_to(out).as_posix())
        if include_originals:
            root = Path(manifest['settings']['source_root'])
            for row in manifest['images']:
                for source in [row] + row['aliases']:
                    path = (root/source['source']).resolve()
                    if not path.is_relative_to(root) or sha256(path) != source['source_sha256']:
                        raise ValueError('Original source changed before export.')
                    z.write(path, 'originals/' + source['source'])
    temporary.replace(archive)
    return archive


## 4. Review the input inventory

Previous depth/normal/albedo previews are excluded by name. Unreadable or animated
images are reported. Exact decoded-pixel duplicates are processed once with every
source alias retained. EXIF orientation is applied; alpha is dropped on RGB
conversion. Original bytes are not modified. TIFFs are converted to a single PNG.


In [ ]:
from IPython.display import display, HTML, Image as DisplayImage, FileLink
import html

sources, skipped = discover(PHOTO_DIR, EXCLUDE_PATTERNS)
known = {s["source"] for s in sources}
unknown = (set(ONLY_RELATIVE_PATHS) | set(EXCLUDE_RELATIVE_PATHS)) - known
if unknown:
    raise ValueError("Selection contains unknown or already excluded paths: " + repr(sorted(unknown)))
selected = [s for s in sources if s["source"] not in EXCLUDE_RELATIVE_PATHS
            and (not ONLY_RELATIVE_PATHS or s["source"] in ONLY_RELATIVE_PATHS)]
skipped += [{"source": s["source"], "reason": "explicit selection"} for s in sources if s not in selected]
print(f"{len(selected)} selected files; {len(skipped)} excluded/unreadable.")
for i, source in enumerate(selected):
    print(f"{i:3d}  {source['source']}  {source['oriented_size']}  EXIF={source['exif_orientation']}")
for source in skipped:
    print("SKIP:", source["source"], "—", source["reason"])
if not selected:
    raise ValueError("No photos selected. Check PHOTO_DIR and filters.")


In [ ]:
manifest = prepare(PHOTO_DIR, OUTPUT_DIR, selected, skipped, MODALITIES, MAX_EDGE, SEED, PROPERTY_ID)
OUTPUT_DIR = OUTPUT_DIR.expanduser().resolve()
print(f"Prepared {len(manifest['images'])} unique photos in {OUTPUT_DIR}")
for row in manifest["images"][:8]:
    print(row["source"], "→", row["input_size"], "aliases:", [a["source"] for a in row["aliases"]])
    display(DisplayImage(filename=str(OUTPUT_DIR / row["input"]), width=260))


## 5. GPU and assets

This check happens before model downloads. Confirm your GPU has enough **free**
VRAM. Models download from Hugging Face; your photos stay on the machine running
this kernel. This notebook downloads only the Qwen transformer/VAE, the chosen
Marigold checkpoints and their precomputed prompt embeddings. All four outputs
currently need about **48.6 GB of weight files**, versus about 75 GB for the broad
upstream downloader, before dependencies, photos, results and caches. Disk and
GPU VRAM are separate limits. The free-space check reads model metadata first;
partial files are conservatively counted as full remaining downloads.

**Recovering from a full disk during the old download:** the next cell first lists
unused model assets and their sizes. It leaves them alone by default. For this
notebook's dedicated asset directory, stop other jobs and set
`REMOVE_UNUSED_ASSETS=True` to remove those listed weights/partial transfers.
Photos, predictions and required weights are kept. Do not use this option if
another workflow needs those components or a prior completed run's asset lock
includes them. You can also run `!python -m pip cache purge` to clear downloaded
package archives (installed packages remain). Avoid deleting the whole Hugging
Face cache or restarting the Colab runtime: you may lose reusable downloads.

A failure before inference can reuse its prepared OUTPUT_DIR. After inference has
recorded code/assets/environment, changes require a new output directory. The new
selective downloader reuses existing required weights. For persistent storage,
set ASSETS_DIR to a mounted disk **before** downloading; changing it does not move
existing weights. The reserve is a headroom estimate, not a guarantee for any
number of photos. Weight fingerprints are read once on first run and verified on
resume, which can take a few minutes.

[Reviewed model loader](https://github.com/huawei-bayerlab/marigold-v2/blob/cc6a7031abcd59fd9e1ceff7fdd0d9687d389bc5/marigoldv2/experiments/20260316_qwen_depth/component_loader.py)
loads the transformer and VAE; the network uses precomputed prompt embeddings.


In [ ]:
unused = prune_unused_assets(ASSETS_DIR, MODALITIES, dry_run=True)
for item in unused:
    print(f"Unused: {item['bytes']/2**30:.2f} GiB  {item['path']}")
print(f"Reclaimable listed assets: {sum(i['bytes'] for i in unused)/2**30:.2f} GiB")
if REMOVE_UNUSED_ASSETS:
    prune_unused_assets(ASSETS_DIR, MODALITIES, dry_run=False)
    print("Removed listed unused assets. Required weights and photos kept.")

import torch
if platform.system() != "Linux" or not torch.cuda.is_available():
    raise RuntimeError("Inference requires a Linux NVIDIA CUDA kernel; preparation can run without it.")
free, total = torch.cuda.mem_get_info()
print(torch.cuda.get_device_name(0), f"free={free/1e9:.1f} GB, total={total/1e9:.1f} GB")
if free < 17e9:
    print("VRAM below upstream's approximate 1024-square requirement. Consider a larger GPU or reduce MAX_EDGE in a new run.")
check_inference_imports(REPO_DIR)
if DOWNLOAD_MODELS:
    download_plan = download_inference_assets(ASSETS_DIR, MODALITIES, reserve_gib=DISK_RESERVE_GIB)
    write_json(OUTPUT_DIR / "download-plan.json", download_plan)
provenance = runtime_provenance(REPO_DIR, ASSETS_DIR, MODALITIES, OUTPUT_DIR)
print("Model/code/environment provenance recorded.")


## 6. Run all selected predictions

The CLI runs at its native-input setting on the prepared, aspect-preserving PNGs.
This avoids fixed-width/height output resizing surprises. Raw arrays align with
`inputs/<id>.png`; the manifest records their relationship to the originals.

The upstream CLI already processes one photo at a time, loading the model once
per modality. Smaller photo groups do not shrink the shared model weights. Staged
inputs use hard links where supported. After a successful modality pass, validated
results are retained and redundant CLI images are removed; logs/configs remain.
Failed attempts remain available for diagnosis.
The notebook prints the last 80 log lines for a failed pass and stops if that
pass yields no validated predictions. Missing NPY files are a consequence;
use the underlying exception in the log to diagnose the failure.

Logs are in each printed attempt folder. Interrupting the cell terminates the
inference process group. Rerun to resume. Partial jobs cannot be exported as a
complete evidence ZIP. Array validation checks file integrity, not geometric accuracy.


In [ ]:
manifest = run_batch(REPO_DIR, ASSETS_DIR, OUTPUT_DIR, provenance,
                     clean_completed_attempts=CLEAN_COMPLETED_ATTEMPTS)
print("Run status:", manifest["status"])
for row in manifest["images"]:
    print(row["source"], "→", list(row["outputs"]), row.get("last_error", ""))
if manifest["status"] != "complete":
    print("Read the attempt logs, resolve the failure (e.g. CUDA memory), and rerun this cell.")


## 7. Inspect arrays and compare previews

Depth values use the selected checkpoint's relative **log-depth** representation,
not metres. Normals use camera coordinates; preserve the raw CHW array and validate
axis conventions before world-space use. Albedo from this pinned CLI is gamma-2.2
converted sRGB. See-through depth is a distinct prediction of surfaces behind glass.

PNG previews are for inspection; never reconstruct raw values from their colours.


In [ ]:
manifest = json.loads((OUTPUT_DIR / "manifest.json").read_text())
PHOTO_INDEX = 0  # Change to inspect another photo.
row = manifest["images"][PHOTO_INDEX]
print(row["source"], "ID:", row["id"])
display(DisplayImage(filename=str(OUTPUT_DIR / row["input"]), width=420))
for modality in MODALITIES:
    record = row["outputs"].get(modality)
    if not record:
        print(modality, "MISSING")
        continue
    array = np.load(OUTPUT_DIR / record["npy"], allow_pickle=False)
    print(modality, array.shape, array.dtype, record["encoding"], record["stats"])
    display(DisplayImage(filename=str(OUTPUT_DIR / record["png"]), width=420))


## 8. Export a reconstruction evidence ZIP

Bring this ZIP back to the reconstruction workspace. It contains originals
(optional), prepared RGBs, raw NPYs, PNG previews, source/alias mappings, input
transforms, checkpoint metadata, runtime versions, hashes, inference logs/configs.
No metric scale or geometry acceptance is implied. ZIP64 supports large batches.

```
manifest.json
asset-lock.json
originals/<source-relative filename>
inputs/<stable image ID>.png
results/<stable image ID>/depth.npy + depth.png
results/<stable image ID>/normals.npy + normals.png
results/<stable image ID>/albedo.npy + albedo.png
results/<stable image ID>/depth_seethrough.npy + depth_seethrough.png
attempts/.../inference.log and config.yaml
```


In [ ]:
archive = export_zip(OUTPUT_DIR, include_originals=INCLUDE_ORIGINALS_IN_ZIP)
print("Evidence archive:", archive)
print(f"Size: {archive.stat().st_size / 1e6:.1f} MB")
# A FileLink works when the archive is inside the Jupyter server's served folder.
# Otherwise use the server file browser / mounted disk to retrieve the printed path.
display(FileLink(str(archive)))
# Optional in Colab: from google.colab import files; files.download(str(archive))
